In [ ]:
!pip install peft==0.8.2
!pip install datasets==2.16.1

In [ ]:
import os
import pandas as pd
import torch
import transformers
from datasets import Dataset
import peft
from peft import LoraConfig, get_peft_model
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    DataCollatorForSeq2Seq
)

# ==========================================
# 1. SETUP MODEL AND TOKENIZER
# ==========================================
model_name = "bigscience/bloom-1b1"

tokenizer = AutoTokenizer.from_pretrained(model_name)
foundation_model = AutoModelForCausalLM.from_pretrained(model_name)

# Move model to GPU
foundation_model.to("cuda")

# ==========================================
# 2. BASE MODEL INFERENCE TEST
# ==========================================
def get_outputs(model, inputs, max_new_tokens=100):
    outputs = model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_new_tokens=max_new_tokens,
        repetition_penalty=1.5,
        early_stopping=True,
        eos_token_id=tokenizer.eos_token_id,
        num_beams=3
    )
    return outputs

input_sentences = tokenizer("How are you?", return_tensors="pt")
input_sentences = {k: v.to("cuda") for k, v in input_sentences.items()}

foundational_outputs_sentence = get_outputs(foundation_model, input_sentences, max_new_tokens=100)
print("Base Model Output:")
print(tokenizer.batch_decode(foundational_outputs_sentence, skip_special_tokens=True))



In [ ]:
custom_prompts = [
    {
        "prompt": "I want you to act as a linux terminal","response":"ਮੈਂ ਚਾਹੁੰਦਾ ਹਾਂ ਕਿ ਤੁਸੀਂ ਇੱਕ ਲਿਨਕਸ ਟਰਮੀਨਲ ਵਜੋਂ ਕੰਮ ਕਰੋ।"
    },
    {
        "prompt": "Explain the concept of machine learning","response":"ਮਸ਼ੀਨ ਲਰਨਿੰਗ ਦੇ ਅਸੂਲ ਨੂੰ ਸਮਝਾਓ।"
    },
    {
        "prompt": "Translate this sentence into Spanish","response":"ਇਹ ਵਾਕ ਨੂੰ ਸਪੈਨਿਸ਼ ਵਿੱਚ ਅਨੁਵਾਦ ਕਰੋ।"
    },
    {
        "prompt": "Create a summary of this text","response":"ਇਸ ਪਾਠ ਦਾ ਇਕ ਸੰਖੇਪ ਬਣਾਓ।"
    },
    {
        "prompt": "Generate a list of 5 healthy foods","response":"5 ਸਿਹਤਮੰਦ ਖਾਣੇ ਦੀ ਸੂਚੀ ਬਣਾਓ।"
    },
    {
        "prompt": "Describe the process of photosynthesis","response":"ਫੋਟੋਸਿੰਥੈਸਿਸ ਦੀ ਪ੍ਰਕਿਰਿਆ ਦਾ ਵਰਣਨ ਕਰੋ।"
    },
    {
        "prompt": "What are the benefits of regular exercise","response":"ਨਿਯਮਿਤ ਵਰਜ਼ਿਸ਼ ਦੇ ਫਾਇਦੇ ਕੀ ਹਨ###"
    },
    {
        "prompt": "Find the capital city of France","response":"ਫ੍ਰਾਂਸ ਦੀ ਰਾਜਧਾਨੀ ਸ਼ਹਿਰ ਲੱਭੋ।"
    },
    {
        "prompt": "How do you solve a quadratic equation","response":"ਤੁਸੀਂ ਇੱਕ ਚਤੁਰਭੁਜ ਸਮੀਕਰਨ ਨੂੰ ਕਿਵੇਂ ਹੱਲ ਕਰਦੇ ਹੋ###"
    },
    {
        "prompt": "List the top 10 programming languages in 2024","response":"2024 ਵਿੱਚ ਸਿਖਰ ਦੇ 10 ਪ੍ਰੋਗ੍ਰਾਮਿੰਗ ਭਾਸ਼ਾਵਾਂ ਦੀ ਸੂਚੀ ਦਿਓ।"
    },
    {
        "prompt": "Write a short story about a talking cat","response":"ਇੱਕ ਬੋਲਦੇ ਬਿੱਲੀ ਬਾਰੇ ਇੱਕ ਛੋਟੀ ਕਹਾਣੀ ਲਿਖੋ।"
    },
    {
        "prompt": "Explain the theory of relativity in simple terms","response":"ਰਿਲੇਟਿਵਿਟੀ ਦੇ ਸਿਧਾਂਤ ਨੂੰ ਸਧਾਰਨ ਸ਼ਬਦਾਂ ਵਿੱਚ ਸਮਝਾਓ।"
    },
    {
        "prompt": "Generate a poem about the moonlight","response":"ਚੰਦਨੀ ਬਾਰੇ ਇੱਕ ਕਵਿਤਾ ਬਣਾਓ।"
    },
    {
        "prompt": "What are some good pickup lines","response":"ਕੁਝ ਵਧੀਆ ਪਿਕਅੱਪ ਲਾਈਨਾਂ ਕੀ ਹਨ###"
    },
    {
        "prompt": "Describe the perfect vacation spot","response":"ਸਬ ਤੋਂ ਵਧੀਆ ਛੁੱਟੀਆਂ ਵਾਲੀ ਥਾਂ ਦਾ ਵਰਣਨ ਕਰੋ।"
    },
    {
        "prompt": "What are the top 10 ways to stay productive","response":"ਉਤਪਾਦਕ ਬਣੇ ਰਹਿਣ ਦੇ ਸਿਖਰ ਦੇ 10 ਤਰੀਕੇ ਕੀ ਹਨ###"
    },
    {
        "prompt": "Create a dialogue between a robot and a human","response":"ਰੋਬੋਟ ਅਤੇ ਮਨੁੱਖ ਦਰਮਿਆਨ ਇੱਕ ਸੰਵਾਦ ਬਣਾਓ।"
    },
    {
        "prompt": "Summarize the plot of 'Hamlet'","response":"'ਹੈਮਲੇਟ' ਦੀ ਕਥਾ ਦਾ ਸੰਖੇਪ ਦਿਓ।"
    },
    {
        "prompt": "Write a motivational speech for students","response":"ਵਿਦਿਆਰਥੀਆਂ ਲਈ ਇੱਕ ਪ੍ਰੇਰਣਾਤਮਕ ਭਾਸ਼ਣ ਲਿਖੋ।"
    },
    {
        "prompt": "Explain the basics of blockchain technology","response":"ਬਲੌਕਚੇਨ ਤਕਨਾਲੋਜੀ ਦੇ ਮੂਲ ਤੱਤ ਸਮਝਾਓ।"
    },
    {
        "prompt": "Describe the lifecycle of a butterfly","response":"ਇੱਕ ਤਿਤਲੀ ਦੀ ਜ਼ਿੰਦਗੀ ਚੱਕਰ ਦਾ ਵਰਣਨ ਕਰੋ।"
    },
    {
        "prompt": "What are the health benefits of yoga","response":"ਯੋਗ ਦੇ ਸਿਹਤ ਲਾਭ ਕੀ ਹਨ###"
    },
    {
        "prompt": "Explain the process of photosynthesis","response":"ਫੋਟੋਸਿੰਥੈਸਿਸ ਦੀ ਪ੍ਰਕਿਰਿਆ ਸਮਝਾਓ।"
    },
    {
        "prompt": "Describe the function of the heart","response":"ਦਿਲ ਦੀ ਕਾਰਜ ਸ਼ੀਲਤਾ ਦਾ ਵਰਣਨ ਕਰੋ।"
    },
    {
        "prompt": "What are the main causes of climate change","response":"ਮੌਸਮ ਪਰੀਵਰਤਨ ਦੇ ਮੁੱਖ ਕਾਰਨ ਕੀ ਹਨ###"
    },
    {
        "prompt": "Create a list of the 7 wonders of the world","response":"ਦੁਨੀਆ ਦੇ 7 ਅਜੂਬਿਆਂ ਦੀ ਸੂਚੀ ਬਣਾਓ।"
    },
    {
        "prompt": "Explain the importance of sleep","response":"ਨੀਂਦ ਦੀ ਮਹੱਤਤਾ ਸਮਝਾਓ।"
    },
    {
        "prompt": "What are the symptoms of the common cold","response":"ਆਮ ਜ਼ੁਕਾਮ ਦੇ ਲੱਛਣ ਕੀ ਹਨ###"
    },
    {
        "prompt": "Describe the process of water filtration","response":"ਪਾਣੀ ਦੇ ਫਿਲਟ੍ਰੇਸ਼ਨ ਦੀ ਪ੍ਰਕਿਰਿਆ ਦਾ ਵਰਣਨ ਕਰੋ।"
    },
    {
        "prompt": "Explain the significance of the Great Wall of China","response":"ਚੀਨ ਦੀ ਮਹਾਨ ਦਿਵਾਰ ਦੀ ਮਹੱਤਤਾ ਸਮਝਾਓ।"
    },
    {
        "prompt": "Generate a creative story about a haunted house","response":"ਇੱਕ ਭੂਤ ਬੰਗਲੇ ਬਾਰੇ ਇੱਕ ਰਚਨਾਤਮਕ ਕਹਾਣੀ ਬਣਾਓ।"
    },
    {
        "prompt": "Describe the key features of a smartphone","response":"ਇੱਕ ਸਮਾਰਟਫੋਨ ਦੀ ਮੁੱਖ ਵਿਸ਼ੇਸ਼ਤਾਵਾਂ ਦਾ ਵਰਣਨ ਕਰੋ।"
    },
    {
        "prompt": "What are the benefits of a balanced diet","response":"ਸੰਤੁਲਿਤ ਖੁਰਾਕ ਦੇ ਫਾਇਦੇ ਕੀ ਹਨ###"
    },
    {
        "prompt": "Explain the water cycle","response":"ਪਾਣੀ ਦੇ ਚੱਕਰ ਨੂੰ ਸਮਝਾਓ।"
    },
    {
        "prompt": "What is the significance of the Taj Mahal","response":"ਤਾਜ ਮਹਲ ਦੀ ਮਹੱਤਤਾ ਕੀ ਹੈ###"
    },
    {
        "prompt": "Generate a list of 10 famous inventors","response":"10 ਮਸ਼ਹੂਰ ਖੋਜਕਾਰਾਂ ਦੀ ਸੂਚੀ ਬਣਾਓ।"
    },
    {
        "prompt": "Describe the process of making chocolate","response":"ਚਾਕਲੇਟ ਬਣਾਉਣ ਦੀ ਪ੍ਰਕਿਰਿਆ ਦਾ ਵਰਣਨ ਕਰੋ।"
    },
    {
        "prompt": "Explain the importance of education","response":"ਸਿੱਖਿਆ ਦੀ ਮਹੱਤਤਾ ਸਮਝਾਓ।"
    },
    {
        "prompt": "What are the different types of renewable energy","response":"ਨਵੀਨੀਕਰਣ ਯੋਗ ਊਰਜਾ ਦੇ ਵੱਖ ਵੱਖ ਤਰੀਕੇ ਕੀ ਹਨ###"
    },
    {
        "prompt": "Describe the anatomy of the human brain","response":"ਮਨੁੱਖੀ ਦਿਮਾਗ ਦੀ ਬਣਾਵਟ ਦਾ ਵਰਣਨ ਕਰੋ।"
    },
    {
        "prompt": "What are the top 5 tourist destinations in India","response":"ਭਾਰਤ ਵਿੱਚ ਸਿਖਰ ਦੇ 5 ਸੈਲਾਨੀ ਮੰਜ਼ਿਲਾਂ ਕੀ ਹਨ###"
    },
    {
        "prompt": "Explain the process of cell division","response":"ਸੈੱਲ ਵਿਭਾਜਨ ਦੀ ਪ੍ਰਕਿਰਿਆ ਸਮਝਾਓ।"
    },
    {
        "prompt": "Describe the role of the Prime Minister in a country","response":"ਕਿਸੇ ਦੇਸ਼ ਵਿੱਚ ਪ੍ਰਧਾਨ ਮੰਤਰੀ ਦੀ ਭੂਮਿਕਾ ਦਾ ਵਰਣਨ ਕਰੋ।"
    },
    {
        "prompt": "What are the benefits of meditation","response":"ਧਿਆਨ ਦੇ ਫਾਇਦੇ ਕੀ ਹਨ###"
    },
    {
        "prompt": "Generate a list of the 10 largest animals on Earth","response":"ਧਰਤੀ ਦੇ 10 ਸਭ ਤੋਂ ਵੱਡੇ ਜਾਨਵਰਾਂ ਦੀ ਸੂਚੀ ਬਣਾਓ।"
    },
    {
        "prompt": "Describe the main events of World War II","response":"ਦੂਜੀ ਵਿਸ਼ਵ ਯੁੱਧ ਦੇ ਮੁੱਖ ਘਟਨਾਵਾਂ ਦਾ ਵਰਣਨ ਕਰੋ।"
    },
    {
        "prompt": "Explain the concept of artificial intelligence","response":"ਕ੍ਰਿਤ੍ਰਿਮ ਬੁੱਧਿਮੱਤਾ ਦੇ ਅਸੂਲ ਨੂੰ ਸਮਝਾਓ।"
    },
    {
        "prompt": "What are the different types of ecosystems","response":"ਵੱਖ ਵੱਖ ਤਰ੍ਹਾਂ ਦੇ ਪਰਿਸ਼ਥਿਤਿਕੀ ਤੰਦਰੁਸਤੀ ਸਿਸਟਮ ਕੀ ਹਨ###"
    },
    {
        "prompt": "Describe the cultural significance of Diwali","response":"ਦਿਵਾਲੀ ਦੀ ਸੱਭਿਆਚਾਰਕ ਮਹੱਤਤਾ ਦਾ ਵਰਣਨ ਕਰੋ।"
    },
    {
        "prompt": "Explain the process of volcanic eruption","response":"ਜਵਾਲਾਮੁਖੀ ਵਿਸਫੋਟ ਦੀ ਪ੍ਰਕਿਰਿਆ ਸਮਝਾਓ।"
    },
    {
        "prompt": "What are the health benefits of drinking water","response":"ਪਾਣੀ ਪੀਣ ਦੇ ਸਿਹਤ ਲਾਭ ਕੀ ਹਨ###"
    },
    {
        "prompt": "Describe the history of the Internet","response":"ਇੰਟਰਨੈਟ ਦੀ ਇਤਿਹਾਸ ਦਾ ਵਰਣਨ ਕਰੋ।"
    },
    {
        "prompt": "Explain the principles of democracy","response":"ਲੋਕਤੰਤਰ ਦੇ ਸਿਧਾਂਤ ਸਮਝਾਓ।"
    },
    {
        "prompt": "What are the main components of a computer","response":"ਕੰਪਿਊਟਰ ਦੇ ਮੁੱਖ ਹਿੱਸੇ ਕੀ ਹਨ###"
    },
    {
        "prompt": "Describe the process of making bread","response":"ਬ੍ਰੇਡ ਬਣਾਉਣ ਦੀ ਪ੍ਰਕਿਰਿਆ ਦਾ ਵਰਣਨ ਕਰੋ।"
    },
    {
        "prompt": "Explain the significance of the Eiffel Tower","response":"ਆਈਫਲ ਟਾਵਰ ਦੀ ਮਹੱਤਤਾ ਸਮਝਾਓ।"
    },
    {
        "prompt": "Generate a list of the top 10 books of all time","response":"ਸਭ ਸਮੇਂ ਦੇ ਸਿਖਰ ਦੇ 10 ਕਿਤਾਬਾਂ ਦੀ ਸੂਚੀ ਬਣਾਓ।"
    },
    {
        "prompt": "Describe the stages of human development","response":"ਮਨੁੱਖੀ ਵਿਕਾਸ ਦੇ ਪੜਾਅ ਦਾ ਵਰਣਨ ਕਰੋ।"
    },
    {
        "prompt": "What are the benefits of learning a second language","response":"ਦੂਜੀ ਭਾਸ਼ਾ ਸਿੱਖਣ ਦੇ ਫਾਇਦੇ ਕੀ ਹਨ###"
    },
    {
        "prompt": "Explain the process of recycling","response":"ਰੀਸਾਈਕਲਿੰਗ ਦੀ ਪ੍ਰਕਿਰਿਆ ਸਮਝਾਓ।"
    },
    {
        "prompt": "Describe the features of the Amazon rainforest","response":"ਅਮੇਜ਼ਨ ਵਰਖਾ ਜੰਗਲ ਦੀਆਂ ਵਿਸ਼ੇਸ਼ਤਾਵਾਂ ਦਾ ਵਰਣਨ ਕਰੋ।"
    },
    {
        "prompt": "What are the top 5 benefits of exercise","response":"ਕਸਰਤ ਦੇ ਸਿਖਰ ਦੇ 5 ਫਾਇਦੇ ਕੀ ਹਨ###"
    },
    {
        "prompt": "Explain the theory of evolution","response":"ਵਿਕਾਸ ਦੇ ਸਿਧਾਂਤ ਨੂੰ ਸਮਝਾਓ।"
    },
    {
        "prompt": "Describe the main functions of the liver","response":"ਜਿਗਰ ਦੀਆਂ ਮੁੱਖ ਕਾਰਜਾਂ ਦਾ ਵਰਣਨ ਕਰੋ।"
    },
    {
        "prompt": "What are the different types of pollution","response":"ਪ੍ਰਦੂਸ਼ਣ ਦੇ ਵੱਖ ਵੱਖ ਤਰੀਕੇ ਕੀ ਹਨ###"
    },
    {
        "prompt": "Explain the significance of the Great Barrier Reef","response":"ਮਹਾਨ ਬੈਰੀਅਰ ਰੀਫ ਦੀ ਮਹੱਤਤਾ ਸਮਝਾਓ।"
    },
    {
        "prompt": "Generate a list of the 10 smallest countries in the world","response":"ਦੁਨੀਆ ਦੇ 10 ਸਭ ਤੋਂ ਛੋਟੇ ਦੇਸ਼ਾਂ ਦੀ ਸੂਚੀ ਬਣਾਓ।"
    },
    {
        "prompt": "Describe the process of fermentation","response":"ਖਮੀਰਕਰਨ ਦੀ ਪ੍ਰਕਿਰਿਆ ਦਾ ਵਰਣਨ ਕਰੋ।"
    },
    {
        "prompt": "What are the benefits of using renewable energy sources","response":"ਨਵੀਨੀਕਰਣ ਯੋਗ ਊਰਜਾ ਸ੍ਰੋਤਾਂ ਦੇ ਫਾਇਦੇ ਕੀ ਹਨ###"
    },
    {
        "prompt": "Explain the importance of biodiversity","response":"ਜੈਵ ਵਿਭਿੰਨਤਾ ਦੀ ਮਹੱਤਤਾ ਸਮਝਾਓ।"
    }
]

In [ ]:
import pandas as pd
from datasets import Dataset

In [ ]:
# ==========================================
# 3. DATA PREPROCESSING (SCENARIO A FIX)
# ==========================================
# (Assumes your custom_prompts list is already loaded above this block)
df = pd.DataFrame(custom_prompts)
dataset = Dataset.from_pandas(df)

def tokenize_and_mask_prompt_response(sample, tokenizer):
    # Separate the text inputs using your dataset's dictionary keys
    prompt_text = sample["prompt"]
    response_text = sample["response"]

    # Tokenize prompt (keeps original tokens intact)
    prompt_ids = tokenizer(prompt_text, add_special_tokens=True)["input_ids"]

    # Tokenize response and cleanly append the EOS token at the end
    response_ids = tokenizer(response_text, add_special_tokens=False)["input_ids"]
    response_ids.append(tokenizer.eos_token_id)

    # Combine inputs
    input_ids = prompt_ids + response_ids
    attention_mask = [1] * len(input_ids)

    # CRITICAL FIX: Mask prompt tokens with -100 so the starting loss is not 0.000000
    labels = [-100] * len(prompt_ids) + response_ids

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels
    }

# Process the dataset and remove old text column names
train_sample = dataset.map(
    tokenize_and_mask_prompt_response,
    remove_columns=dataset.column_names,
    fn_kwargs={"tokenizer": tokenizer}
)

# Display the newly formatted tokenized sample
print("\nTokenized Dataset Info:")
print(train_sample)
print("\nFirst Processed Row Sample:")
print(train_sample[0])

In [ ]:
# ==========================================
# 4. PEFT / LORA CONFIGURATION
# ==========================================
lora_config = LoraConfig(
    r=4,
    lora_alpha=1,
    target_modules=["query_key_value"], # Targets BLOOM's attention layer names
    lora_dropout=0.05,
    bias="lora_only",
    task_type="CAUSAL_LM"
)

peft_model = get_peft_model(foundation_model, lora_config)
print("\nTrainable Parameters:")
peft_model.print_trainable_parameters()

# ==========================================
# 5. OPTIMIZED TRAINING CONFIGURATION
# ==========================================
working_dir = './'
output_directory = os.path.join(working_dir, "peft_punjabi_outputs")

training_args = TrainingArguments(
    output_dir=output_directory,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=16,
    # CRITICAL FIX: Lowered learning rate from 3e-3 to prevent NaN/weight explosion
    learning_rate=2e-4,
    num_train_epochs=10, # Lowered from 100 to prevent overfitting
    bf16=False,
    # CRITICAL FIX: Changed from 16 to 1 to immediately view step 1 loss metrics
    logging_steps=1,
)

# ==========================================
# 6. INITIALIZE AND LAUNCH TRAINER
# ==========================================
trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=train_sample,
    # Sequence-to-Sequence collator handles dynamic token padding for custom labels securely
    data_collator=DataCollatorForSeq2Seq(
        tokenizer,
        pad_to_multiple_of=8,
        return_tensors="pt",
        padding=True
    )
)

print("\nStarting LoRA Training...")
trainer.train()
